# Анализ и визуализация данных о Нобелевских лауреатах

В этом ноутбуке анализируются два CSV-файла:

- `nobel_laureates.csv` — данные о людях, получивших Нобелевскую премию;
- `nobel_awards.csv` — данные о награждениях Нобелевской премией.

Цель анализа — построить графики по категориям премии, годам награждения, странам, полу лауреатов, типу получателя и роду занятий.


## 1. Импорт библиотек

In [ ]:
# подключаем библиотеки для таблиц и графиков
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# настраиваем внешний вид графиков
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)


## 2. Загрузка данных из GitHub

In [ ]:
# указываем владельца и название репозитория
repo_owner = "Nikto555"
repo_name = "python-ai-Sorokin-Kirill"

# собираем базовую ссылку на папку data
base_url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/main/data"

# указываем raw-ссылки на csv-файлы
laureates_url = f"{base_url}/nobel_laureates.csv"
awards_url = f"{base_url}/nobel_awards.csv"

# читаем csv-файлы
df_laureates = pd.read_csv(laureates_url)
df_awards = pd.read_csv(awards_url)

# выводим размеры таблиц
print(f"nobel_laureates.csv: {df_laureates.shape[0]} строк, {df_laureates.shape[1]} столбцов")
print(f"nobel_awards.csv: {df_awards.shape[0]} строк, {df_awards.shape[1]} столбцов")

# показываем исходные столбцы
print("Столбцы nobel_laureates.csv:")
print(df_laureates.columns.tolist())

print("\nСтолбцы nobel_awards.csv:")
print(df_awards.columns.tolist())

# показываем первые строки
display(df_laureates.head())
display(df_awards.head())


## 3. Подготовка данных

In [ ]:
# переименовываем столбцы в таблице лауреатов
df_laureates = df_laureates.rename(columns={
    "laureate": "laureate_url",
    "laureateLabel": "laureate",
    "prize": "prize_url",
    "prizeLabel": "prize",
    "awardDate": "award_date",
    "awardYear": "award_year",
    "citizenshipLabel": "citizenship",
    "genderLabel": "gender",
    "birthDate": "birth_date",
    "birthYear": "birth_year",
    "occupationLabel": "occupation"
})

# переименовываем столбцы в таблице награждений
df_awards = df_awards.rename(columns={
    "recipient": "recipient_url",
    "recipientLabel": "recipient",
    "recipientType": "recipient_type",
    "prize": "prize_url",
    "prizeLabel": "prize",
    "awardDate": "award_date",
    "awardYear": "award_year",
    "countryLabel": "country"
})

# создаём недостающие столбцы, чтобы ноутбук не падал при неполных данных
required_laureate_columns = [
    "laureate", "prize", "award_date", "award_year",
    "citizenship", "gender", "birth_date", "birth_year", "occupation"
]

required_award_columns = [
    "recipient", "recipient_type", "prize", "award_date", "award_year", "country"
]

for column in required_laureate_columns:
    if column not in df_laureates.columns:
        df_laureates[column] = pd.NA

for column in required_award_columns:
    if column not in df_awards.columns:
        df_awards[column] = pd.NA

# преобразуем годы к числовому типу
df_laureates["award_year"] = pd.to_numeric(df_laureates["award_year"], errors="coerce")
df_laureates["birth_year"] = pd.to_numeric(df_laureates["birth_year"], errors="coerce")
df_awards["award_year"] = pd.to_numeric(df_awards["award_year"], errors="coerce")

# преобразуем даты к datetime
df_laureates["award_date"] = pd.to_datetime(df_laureates["award_date"], errors="coerce", utc=True)
df_laureates["birth_date"] = pd.to_datetime(df_laureates["birth_date"], errors="coerce", utc=True)
df_awards["award_date"] = pd.to_datetime(df_awards["award_date"], errors="coerce", utc=True)

# если год не прочитался, пробуем взять его из даты награждения
df_laureates["award_year"] = df_laureates["award_year"].fillna(df_laureates["award_date"].dt.year)
df_awards["award_year"] = df_awards["award_year"].fillna(df_awards["award_date"].dt.year)

# рассчитываем возраст лауреата при награждении
df_laureates["age_at_award"] = df_laureates["award_year"] - df_laureates["birth_year"]

# проверяем результат
display(df_laureates.head())
display(df_awards.head())


## 4. Проверка пропусков

In [ ]:
# проверяем пропуски
print("Пропуски в nobel_laureates:")
display(df_laureates.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_count"}))

print("\nПропуски в nobel_awards:")
display(df_awards.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_count"}))


## График 1. Количество награждений по категориям Нобелевской премии

In [ ]:
# считаем количество награждений по категориям
prize_counts = (
    df_awards["prize"]
    .dropna()
    .value_counts()
    .reset_index()
)
prize_counts.columns = ["prize", "award_count"]

if prize_counts.empty:
    print("Нет данных для графика по категориям премии.")
else:
    plt.figure(figsize=(11, 6))
    sns.barplot(data=prize_counts, x="award_count", y="prize")

    plt.title("Количество награждений по категориям Нобелевской премии")
    plt.xlabel("Количество награждений")
    plt.ylabel("Категория премии")

    plt.tight_layout()
    plt.show()


**Вывод:**  
График показывает, как награждения распределены между разными категориями Нобелевской премии. Это позволяет сравнить представленность научных, общественных и литературных направлений.


## График 2. Динамика награждений по годам

In [ ]:
# считаем количество награждений по годам
awards_by_year = (
    df_awards
    .dropna(subset=["award_year"])
    .groupby("award_year")
    .size()
    .reset_index(name="award_count")
)

# оставляем разумный диапазон лет
awards_by_year = awards_by_year[
    (awards_by_year["award_year"] >= 1900) &
    (awards_by_year["award_year"] <= 2026)
]

if awards_by_year.empty:
    print("Нет данных для графика динамики по годам.")
else:
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=awards_by_year, x="award_year", y="award_count", marker="o")

    plt.title("Динамика количества Нобелевских награждений по годам")
    plt.xlabel("Год")
    plt.ylabel("Количество награждений")

    plt.tight_layout()
    plt.show()


**Вывод:**  
Линейный график показывает изменение количества награждений по годам. Он помогает увидеть историческую динамику и годы с большим или меньшим числом наград.


## График 3. Топ-10 стран по количеству лауреатов

In [ ]:
# считаем страны гражданства лауреатов
top_countries = (
    df_laureates["citizenship"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)
top_countries.columns = ["citizenship", "laureate_count"]

if top_countries.empty:
    print("Нет данных для графика по странам.")
else:
    plt.figure(figsize=(11, 6))
    sns.barplot(data=top_countries, x="laureate_count", y="citizenship")

    plt.title("Топ-10 стран по количеству Нобелевских лауреатов")
    plt.xlabel("Количество лауреатов")
    plt.ylabel("Страна гражданства")

    plt.tight_layout()
    plt.show()


**Вывод:**  
График показывает, какие страны чаще всего встречаются среди гражданств лауреатов. Это помогает анализировать географическое распределение Нобелевских премий.


## График 4. Распределение лауреатов по полу

In [ ]:
# считаем распределение лауреатов по полу
gender_counts = (
    df_laureates["gender"]
    .dropna()
    .value_counts()
    .reset_index()
)
gender_counts.columns = ["gender", "laureate_count"]

if gender_counts.empty:
    print("Нет данных для графика по полу.")
else:
    plt.figure(figsize=(9, 5))
    sns.barplot(data=gender_counts, x="gender", y="laureate_count")

    plt.title("Распределение Нобелевских лауреатов по полу")
    plt.xlabel("Пол")
    plt.ylabel("Количество лауреатов")

    plt.tight_layout()
    plt.show()


**Вывод:**  
График показывает соотношение лауреатов по полу. Такой анализ помогает увидеть, насколько равномерно представлены разные группы среди получателей премии.


## График 5. Тип получателя премии

In [ ]:
# считаем типы получателей
recipient_type_counts = (
    df_awards["recipient_type"]
    .dropna()
    .value_counts()
    .reset_index()
)
recipient_type_counts.columns = ["recipient_type", "award_count"]

if recipient_type_counts.empty:
    print("Нет данных для графика по типу получателя.")
else:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=recipient_type_counts, x="recipient_type", y="award_count")

    plt.title("Распределение по типу получателя премии")
    plt.xlabel("Тип получателя")
    plt.ylabel("Количество награждений")

    plt.tight_layout()
    plt.show()


**Вывод:**  
График показывает, кто чаще получает Нобелевскую премию: отдельные люди или организации. Большинство награждений связано с отдельными лауреатами.


## График 6. Топ-10 родов занятий среди лауреатов

In [ ]:
# считаем самые частые роды занятий
occupation_counts = (
    df_laureates["occupation"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)
occupation_counts.columns = ["occupation", "laureate_count"]

if occupation_counts.empty:
    print("Нет данных для графика по родам занятий.")
else:
    plt.figure(figsize=(11, 6))
    sns.barplot(data=occupation_counts, x="laureate_count", y="occupation")

    plt.title("Топ-10 родов занятий среди Нобелевских лауреатов")
    plt.xlabel("Количество лауреатов")
    plt.ylabel("Род занятий")

    plt.tight_layout()
    plt.show()


**Вывод:**  
График показывает, какие профессии и сферы деятельности чаще встречаются среди лауреатов. Это помогает понять, какие научные и общественные роли наиболее представлены в данных.


## График 7. Возраст лауреатов при награждении

In [ ]:
# оставляем реалистичный диапазон возраста
age_data = df_laureates[
    (df_laureates["age_at_award"] >= 15) &
    (df_laureates["age_at_award"] <= 100)
]["age_at_award"].dropna()

if age_data.empty:
    print("Нет данных для графика возраста лауреатов.")
else:
    plt.figure(figsize=(11, 6))
    sns.histplot(age_data, bins=25, kde=True)

    plt.title("Распределение возраста лауреатов при награждении")
    plt.xlabel("Возраст при награждении")
    plt.ylabel("Количество лауреатов")

    plt.tight_layout()
    plt.show()


**Вывод:**  
Этот график показывает, в каком возрасте лауреаты чаще получают Нобелевскую премию. Он связывает год рождения и год награждения и добавляет к анализу биографический аспект.


## Общий вывод

В этом ноутбуке были проанализированы данные о Нобелевских лауреатах и награждениях.

Графики показали распределение премий по категориям, динамику награждений по годам, страны с наибольшим количеством лауреатов, распределение по полу, типы получателей и наиболее частые роды занятий.

Данные подходят для анализа, потому что содержат как категориальные признаки, например премию, страну, пол, тип получателя и род занятий, так и числовые признаки, например год награждения, год рождения и возраст лауреата при награждении.

В целом анализ помогает увидеть, как Нобелевские премии распределяются по странам, областям знания и биографическим характеристикам лауреатов.
